# DeepSWE on Kubernetes: run, watch, understand

Run 10 [DeepSWE](https://hub.harborframework.com/datasets) coding tasks against a deployed NeMo Helix, watch them run in parallel, then see which tasks the agent solved, where its time went, and why it failed.

- **What runs where:** one Kubernetes Job per task. Each Job runs the agent in an OpenSandbox sandbox, scores it with the task's hidden tests in a second sandbox, and publishes the trial to an Intake Experiment. The run controller merges the 10 results into one bundle.
- **Time:** all 10 tasks run at once, so the slowest sets the pace: up to about 40 min with `codex` (most tasks take 6–14 min; `fastapi-implicit-head-options` took 16–34 min in our runs), about 2 min with `oracle`.
- **You need:** the `nemo` CLI (`pip install nemo-helix`) in this kernel's environment, a CLI context for the deployment (`nemo config set --context <name> --base-url <url>`), and a model the deployment's Inference Gateway serves.
- **Your admin sets the sandbox profiles** (network policy, CPU/memory ceilings) in the Helm values `evals.runs.harbor.sandbox_profiles`. Codex installs itself in the sandbox, so the profile you run under must list package hosts in `agent_setup_egress`. An admin can change a profile or add one; you pick it with `SANDBOX_PROFILE`.

Every action below is a `nemo` command, printed before it runs, so you can repeat it in a terminal.

## 0. Parameters

Change these and run all cells. Re-running cells in the same kernel returns the run already started; a new kernel starts a new run. Set `RUN_ID` to reopen an earlier one.

In [ ]:
# Parameters: the deployment, the tasks, the agent and the run
CONTEXT = "<your-context>"  # your deployment's CLI context: `nemo config view --all-contexts`
WORKSPACE = "default"
DATASET = "datacurve/deep-swe@sha256:aaa82ceb8404dccc17689c9383f93dbcbc8f029a7601d2e3856a416f2cb89269"
TASKS = [  # any of DeepSWE's 113 tasks: `nemo evals runs submit --harbor-hub <DATASET> --list-tasks`
    "anko-default-function-arguments",  # go
    "geo-shapeindex-serialization",  # go
    "expr-try-catch-errors",  # go
    "fastapi-implicit-head-options",  # python
    "mashumaro-flattened-dataclass-fields",  # python
    "dateutil-rfc5545-timezone-interop",  # python
    "happy-dom-abort-pending-body-reads",  # typescript
    "awilix-async-container-initialization",  # typescript
    "csstree-shorthand-expansion-compression",  # javascript
    "fd-deterministic-multi-key-sorting",  # rust
]
MODE = "taskset"  # "taskset": publish the tasks, build images, then submit (3 commands) | "submit": all in 1 command
AGENT = "codex"  # "codex" calls MODEL | "oracle" applies each task's reference solution (no model, checks the setup)
MODEL = "default/azure-openai-gpt-6-luna"  # replace with a model your gateway serves; codex calls it through a closed-book model
CODEX_VERSION = "0.162.1"
EXPERIMENT = "deepswe-demo"  # Intake Experiment: created if missing; each run is one row of its leaderboard
SANDBOX_PROFILE: str | None = (
    None  # None: the deployment's default profile; otherwise a name from the admin's Helm values
)
INFERENCE_KEY_SECRET: str | None = (
    None  # only when platform auth is on: a secret holding a Scoped Access Key (--scope inference)
)
RUN_ID: str | None = None  # set to an existing run id to skip submitting and only analyze it
COMPARE: list[str] = []  # 2+ run ids to compare in section 7, e.g. an oracle run and a codex run
POLL_SECONDS = 15

## 1. Connect and prepare

Installs pandas and matplotlib into the kernel if missing, checks the deployment answers, then creates what the run needs, once:

- the **Intake Experiment** `EXPERIMENT`: every submit passes `--experiment`, so the run gets an Evaluation there at submit and each task's trial lands in it as the task ends.
- for `codex`, a **closed-book virtual model** in front of `MODEL`. DeepSWE tasks have no network, so the platform takes a model-calling agent only through a model whose middleware strips provider-side tools such as web search.

In [ ]:
# Check the kernel has NeMo Helix's evals plugin; install pandas and matplotlib if missing
import importlib
import importlib.util
import shutil
import subprocess
import sys

# Stop early if this kernel can't run `nemo evals`
print(f"Kernel: {sys.executable}")
if importlib.util.find_spec("nemo_evals") is None:
    raise ModuleNotFoundError(
        "This kernel has no NeMo Helix evals plugin. Pick the environment where `nemo evals` works as the kernel, "
        "or install it into this one: pip install 'nemo-helix[nemo-evals-plugin]'"
    )
# Install what's missing with pip, or with uv when the venv has no pip
missing = [name for name in ("pandas", "matplotlib") if importlib.util.find_spec(name) is None]
if missing:
    if importlib.util.find_spec("pip"):
        installer = [sys.executable, "-m", "pip", "install"]
    elif shutil.which("uv"):
        installer = ["uv", "pip", "install", "--python", sys.executable]  # uv-made venvs have no pip
    else:
        raise RuntimeError(f"Neither pip nor uv is available to install {missing} into {sys.executable}")
    print("$ " + " ".join([*installer, *missing]))
    subprocess.run([*installer, *missing], check=True)
    importlib.invalidate_caches()  # so the next cell imports what was just installed

In [ ]:
# Imports and the `nemo` CLI helper; check the deployment, create the Experiment and closed-book model if missing
import hashlib
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
import tomllib
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Markdown, display
from nemo_helix_plugin.client.client import NemoClient

# The CLI, its environment (CONTEXT picks the deployment), the output folder
NEMO = shutil.which("nemo") or str(Path(sys.executable).parent / "nemo")
ENV = {k: v for k, v in os.environ.items() if k not in ("NHX_BASE_URL", "NHX_CURRENT_CONTEXT")}  # CONTEXT decides
WORK = Path("deepswe-demo-output").resolve()
WORK.mkdir(exist_ok=True)


# Run a `nemo` command against CONTEXT, with retries
def nemo(*args, as_json=False, stream=False, quiet=False, attempts=1, timeout=None):
    """Run `nemo --context CONTEXT <args>`; print the command, raise on failure, return stdout (or parsed JSON).

    `attempts` > 1 retries a read every 10 s, for connections that drop (e.g. a port-forward reconnecting).
    `timeout` (s) ends a call that hangs; it then counts as a failed attempt.
    """
    for attempt in range(1, attempts + 1):
        try:
            return _nemo_once(args, as_json=as_json, stream=stream, quiet=quiet or attempt > 1, timeout=timeout)
        except RuntimeError as exc:
            if attempt == attempts:
                raise
            print(f"{exc}; retrying in 10 s ({attempt}/{attempts - 1})")
            time.sleep(10)


# One attempt: stream or capture the output; raise with the CLI's error
def _nemo_once(args, *, as_json, stream, quiet, timeout):
    cmd = [NEMO, "--context", CONTEXT, *args]
    if not quiet:
        print("$ " + shlex.join(["nemo", *cmd[1:]]))
    if stream:  # print each line as it arrives: image builds and submit report progress this way
        proc = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        assert proc.stdout is not None  # stdout=PIPE
        lines = []
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        out, code, err = "".join(lines), proc.wait(), ""
    else:
        try:
            done = subprocess.run(cmd, env=ENV, capture_output=True, text=True, timeout=timeout)
        except subprocess.TimeoutExpired:
            raise RuntimeError(f"nemo {args[0]} {args[1]} timed out after {timeout} s") from None
        out, code, err = done.stdout, done.returncode, done.stderr
    if code:
        raise RuntimeError(f"nemo {args[0]} {args[1]} failed ({code}): {(err or out).strip()[-1500:]}")
    return json.loads(out) if as_json else out


# Whether a resource exists
def exists(*args):
    """True when a `nemo … get` command finds the resource."""
    try:
        nemo(*args, quiet=True)
        return True
    except RuntimeError:
        return False


# Check the deployment answers
if CONTEXT.startswith("<"):
    raise ValueError("Set CONTEXT in section 0 to your deployment's CLI context (`nemo config view --all-contexts`).")
BASE_URL = NemoClient.from_config(context=CONTEXT).base_url.rstrip("/")
with urllib.request.urlopen(f"{BASE_URL}/health/ready", timeout=10) as response:
    print(f"{CONTEXT}: {BASE_URL} is {json.load(response)['status']}")

# Create the Intake Experiment if missing
if not RUN_ID and not exists("experiments", "get", EXPERIMENT, "--workspace", WORKSPACE):  # RUN_ID: analysis only
    nemo("experiments", "create", EXPERIMENT, "--workspace", WORKSPACE, "--description", "DeepSWE demo runs")

# For a model-calling agent: create the closed-book virtual model if missing
CLOSED_BOOK = f"{MODEL.split('/')[-1]}-closed-book"
if (
    not RUN_ID
    and AGENT != "oracle"
    and not exists("inference", "virtual-models", "get", CLOSED_BOOK, "--workspace", WORKSPACE)
):
    middleware = [{"name": "nemo-evals-closed-book", "config_type": "closed_book", "config": {}}]
    nemo(
        "inference",
        "virtual-models",
        "create",
        CLOSED_BOOK,
        "--workspace",
        WORKSPACE,
        "--default-model-entity",
        MODEL,
        "--request-middleware",
        json.dumps(middleware),
    )
# Report what's ready, with the Experiment's Studio link
EXPERIMENT_URL = f"{BASE_URL}/studio/workspaces/{WORKSPACE}/experiment/{EXPERIMENT}"
print(
    f"Experiment {WORKSPACE}/{EXPERIMENT} ready"
    + ("" if AGENT == "oracle" else f"; model {WORKSPACE}/{CLOSED_BOOK} ready")
)
print(f"View the experiment in Studio: {EXPERIMENT_URL}")

The run request: the agent, its model, and where task Jobs run. `--harbor-hub` or `--taskset` supplies the tasks.

In [ ]:
# The run spec (agent, model, sandbox profile) and its idempotency key
# The agent: oracle, or codex calling MODEL through the closed-book model
if AGENT == "oracle":
    target = {"kind": "harbor", "source": {"name": "oracle"}}
else:
    target = {
        "kind": "harbor",
        "source": {"name": AGENT, "model": {"ref": f"{WORKSPACE}/{CLOSED_BOOK}"}},  # evals sets the agent's base URL
        "agent_kwargs": {"version": CODEX_VERSION},
    }
    if INFERENCE_KEY_SECRET:
        target["env_secrets"] = {"OPENAI_API_KEY": INFERENCE_KEY_SECRET}
# The run spec: the agent, plus where task Jobs and sandboxes run
spec = {
    # max_retries: a trial that ends in an error (e.g. a failed codex install) reruns once in a new sandbox
    "spec": {"target": target | {"n_attempts": 1, "max_retries": 1}},
    "execution": {"dispatcher_profile": "default"}
    | ({"sandbox": {"profile": SANDBOX_PROFILE}} if SANDBOX_PROFILE else {}),
}
# Same request in the same kernel, same key: the API returns the run it already started instead of another.
SESSION = globals().get("SESSION") or os.urandom(4).hex()  # once per kernel: a new kernel starts a new run
KEY = (
    "deepswe-demo-"
    + hashlib.sha256(json.dumps([DATASET, sorted(TASKS), spec, MODE, EXPERIMENT, SESSION]).encode()).hexdigest()[:20]
)
# Write the spec to a file named by the key, for `runs submit --spec-file`
SPEC_FILE = WORK / f"{KEY}.json"
SPEC_FILE.write_text(json.dumps(spec, indent=2))
print(SPEC_FILE.read_text())

## 2. Meet the tasks

Each DeepSWE task is a feature request or bug fix in a real open-source repository. The agent gets the instruction and the repository; hidden tests decide the reward (1 = solved).

In [ ]:
# Each task's title, language, category and repo, from its task.toml on Harbor Hub
from nemo_evals.runs_cli import download_hub_tasks  # the same Harbor Hub download `--harbor-hub` uses


# Download the tasks' task.toml files and tabulate their metadata
def task_metadata(names):
    """Title, language, category and repo of DeepSWE tasks, from their task.toml on Harbor Hub."""
    with tempfile.TemporaryDirectory() as scratch, ThreadPoolExecutor(1) as pool:
        # In a thread: the download runs its own event loop, and Jupyter's is already running.
        _, _, task_dirs = pool.submit(download_hub_tasks, DATASET, ",".join(names), Path(scratch)).result()
        tomls = {d.name: tomllib.loads((d / "task.toml").read_text())["metadata"] for d in task_dirs}
    return pd.DataFrame(
        [
            {
                "task": name,
                "title": m.get("display_title"),
                "language": m.get("language"),
                "category": m.get("category"),
                "repo": m.get("repository_url", "").removeprefix("https://github.com/"),
            }
            for name, m in tomls.items()
        ]
    ).set_index("task")


# Show the tasks
meta = task_metadata(TASKS)
display(
    meta.style.set_properties(**{"text-align": "left"}).set_table_styles(
        [{"selector": "th", "props": "text-align: left;"}]
    )
)

## 3. Submit

- **`MODE = "taskset"`** splits the work into 3 commands: `tasksets create` publishes the tasks as a versioned taskset, `tasksets build-images` copies each task's image into the platform's registry, `runs submit --taskset` starts the run. Keep the taskset to run other agents or models on the same tasks without publishing or building again.
- **`MODE = "submit"`** does all three in `runs submit --harbor-hub`.

Interrupting a wait stops only the notebook: builds and runs keep going, and re-running the cell picks them up.

In [ ]:
# Submit the run: create a taskset, build its images, submit ("taskset"), or one `runs submit --harbor-hub` ("submit")
# `runs submit` with this notebook's spec, Experiment and key
def submit(*args):
    """`runs submit` with the run settings; retries once on public ECR's 429. Returns the run id."""
    cmd = [
        "evals",
        "runs",
        "submit",
        *args,
        "--spec-file",
        str(SPEC_FILE),
        "--experiment",
        EXPERIMENT,
        "--idempotency-key",
        KEY,
        "--workspace",
        WORKSPACE,
    ]
    for attempt in (1, 2):
        try:
            out = nemo(*cmd, stream=True)
            match = re.search(r"^Run (run-[0-9a-f]+)", out, re.M)
            if match is None:
                raise RuntimeError("no run id in the submit output")
            return match.group(1)
        except RuntimeError as exc:
            if attempt == 2 or "429" not in str(exc):
                raise
            print("Public ECR rate-limited the image lookup; retrying in 60 s with the same key")
            time.sleep(60)


# RUN_ID set: skip submitting; otherwise submit as MODE says
if RUN_ID:
    print(f"Analyzing existing run {RUN_ID}")
elif MODE == "taskset":
    created = nemo(
        "evals",
        "tasksets",
        "create",
        "--harbor-hub",
        DATASET,
        "--tasks",
        ",".join(TASKS),
        "--workspace",
        WORKSPACE,
        "--json",
        as_json=True,
    )
    TASKSET = created["taskset_ref"]
    print(f"Taskset {TASKSET} ({len(created['tasks'])} of {created['total']} tasks)")
    nemo("evals", "tasksets", "build-images", TASKSET, "--wait", stream=True)
    RUN_ID = submit("--taskset", TASKSET, "--dataset", created["dataset"])
elif MODE == "submit":
    RUN_ID = submit("--harbor-hub", DATASET, "--tasks", ",".join(TASKS))
else:
    raise ValueError(f'MODE is "taskset" or "submit", not {MODE!r}')

## 4. Watch it run

One tile per task, refreshed every `POLL_SECONDS`: grey = waiting, blue = running, green = finished (pass or fail shows in section 5), red = failed to run, amber = expired. The header names the tasks still running: the run takes as long as its slowest task. The cell returns once the controller has merged the results.

In [ ]:
# Poll the run's status and redraw one tile per task until the results are merged

# Each task state's tile color and label; pass or fail shows in section 5
STYLE = {
    "image_preparing": ("#9e9e9e", "preparing image"),
    "queued": ("#9e9e9e", "queued"),
    "registered": ("#0072B2", "running"),
    "succeeded": ("#009E73", "finished"),
    "failed": ("#D55E00", "failed to run"),
    "cancelled": ("#9e9e9e", "cancelled"),
    "expired": ("#E69F00", "expired"),
}
JOB_LABELS = {"created": "starting", "pending": "starting", "completed": "collecting results"}  # a running task's Job
DONE = {"succeeded", "failed", "cancelled", "expired"}
TERMINAL = {"completed", "failed", "cancelled", "expired"}


# A task's name without the dataset prefix
def short(task):
    return task.split("/")[-1].removeprefix(DATASET.split("/")[1].split("@")[0] + "-")


# The board: a header that says what the run is waiting for, then one tile per task
def board(status):
    run, tasks = status["run"], status["tasks"]
    started = datetime.fromisoformat(run["accepted_at"])
    elapsed = int((datetime.now(started.tzinfo) - started).total_seconds())
    waiting = [short(t["task"]) for t in tasks if t["state"].split(" ")[0] not in DONE]
    if run["phase"] in TERMINAL:
        progress = f"run {run['phase']} · result {run['result_state']} · submitted {started:%Y-%m-%d %H:%M} UTC"
    elif waiting:
        progress = (
            f"{len(tasks) - len(waiting)} of {len(tasks)} tasks done · {elapsed // 60}m{elapsed % 60:02d}s since submit"
            f" · waiting on: {', '.join(waiting[:3])}" + (f" and {len(waiting) - 3} more" if len(waiting) > 3 else "")
        )
    else:
        bundle = status.get("bundle") or {}
        progress = f"all {len(tasks)} tasks done · merging results" + (
            f" (Job {bundle['job']}, {bundle['status']})" if bundle.get("job") else ""
        )
    tiles = ""
    for t in tasks:
        phase, name = t["state"].split(" ")[0], short(t["task"])
        color, label = STYLE.get(phase, ("#9e9e9e", phase))
        if phase == "registered":
            label = JOB_LABELS.get(t.get("job_status") or "", label)
        tiles += (
            f'<div style="background:{color};color:white;border-radius:8px;padding:8px 10px;font:12px sans-serif">'
            f"<b>{name}</b><br>{meta.language.get(name, '')} · {label}</div>"
        )
    return HTML(
        f'<div style="font:13px sans-serif;margin-bottom:6px"><b>{run["run_id"]}</b> · {progress}</div>'
        f'<div style="display:grid;grid-template-columns:repeat(5,1fr);gap:6px">{tiles}</div>'
    )


# Poll until the run is over and its result merged; a hung call times out and is retried
print(
    f"$ nemo --context {CONTEXT} evals runs status {RUN_ID} --workspace {WORKSPACE} --json   # every {POLL_SECONDS} s"
)
view = display(HTML("waiting for the first status…"), display_id=True)
try:
    while True:
        status = nemo(
            "evals",
            "runs",
            "status",
            RUN_ID,
            "--workspace",
            WORKSPACE,
            "--json",
            as_json=True,
            quiet=True,
            attempts=6,
            timeout=60,
        )
        view.update(board(status))
        if status["run"]["phase"] in TERMINAL and status["run"]["result_state"] != "pending":
            break
        time.sleep(POLL_SECONDS)
except KeyboardInterrupt:
    print("Stopped watching; the run keeps going. Re-run this cell to pick it up again.")
print(
    f"Run {RUN_ID}: {status['run']['phase']}, result {status['run']['result_state']}"
    + (f" ({status['run']['error']})" if status["run"].get("error") else "")
)

## 5. Results

Downloads the merged bundle (every trial and score) and each task's own bundle (Harbor's phase timings, the agent's trajectory, the hidden tests' output), then joins them with the task metadata.

In [ ]:
# Download the merged bundle and each task's bundle; build the per-task and per-phase tables
# Read a JSON Lines file
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


# The merged bundle, once the run's result is ready
OUT = WORK / RUN_ID
bundle = OUT / "agent-eval-results"  # the merged bundle: only when the run's result is ready
if status["run"]["result_state"] == "ready" and not bundle.exists():
    nemo("evals", "runs", "download", RUN_ID, "--workspace", WORKSPACE, "-o", str(OUT), attempts=3)

# Each task's bundle: its trial, reward, phase times, trajectory and test output
rows, phases, details, run_meta = [], [], {}, {}
for task in status["tasks"]:
    folder = OUT / "tasks" / str(task.get("job"))
    if task.get("job_status") == "completed" and not folder.exists():  # only a completed Job has results
        archive = OUT / f"{task['job']}.tar.gz"
        nemo(
            "jobs",
            "results",
            "download",
            "agent-eval-results",
            "--job",
            task["job"],
            "--workspace",
            WORKSPACE,
            "-o",
            str(archive),
            quiet=True,
            attempts=3,
        )
        with tarfile.open(archive) as tar:
            tar.extractall(folder, filter="data")
        archive.unlink()
    task_bundle = folder / "agent-eval-results"
    if not (task_bundle / "trials.jsonl").exists():  # no trial: the run's own record says why
        error = task.get("error") or task["state"]
        rows.append({"task": short(task["task"]), "reward": None, "error": error.split(":")[0], "error_message": error})
        continue
    run_meta = run_meta or json.loads((task_bundle / "metadata.json").read_text())
    rewards = {
        s["trial_id"]: next((o["value"] for o in s["outputs"] if o["name"] == "reward"), None)
        for s in read_jsonl(task_bundle / "scores.jsonl")
        if s["metric_type"] == "harbor_reward"
    }
    for trial in read_jsonl(task_bundle / "trials.jsonl"):
        m, error = trial["measurements"], trial.get("error") or {}
        rows.append(
            {
                "task": trial["task_id"].split("/")[-1],
                "reward": rewards.get(trial["id"]),
                "error": error.get("type"),
                "error_message": error.get("message"),
                "tokens": m.get("total_tokens"),
                "cost_usd": m.get("cost_usd"),
                "agent_sec": m.get("runtime_sec"),
            }
        )
    for result_file in task_bundle.glob("evidence/harbor/*/*__*/result.json"):
        r, trial_dir = json.loads(result_file.read_text()), result_file.parent
        name = r["task_name"].split("/")[-1]
        for phase in ("environment_setup", "agent_setup", "agent_execution", "verifier"):
            if r.get(phase):
                phases.append(
                    {
                        "task": name,
                        "phase": phase,
                        "start": pd.Timestamp(r[phase]["started_at"]),
                        "end": pd.Timestamp(r[phase]["finished_at"]),
                    }
                )
        trajectory = trial_dir / "agent" / "trajectory.json"
        steps = json.loads(trajectory.read_text())["steps"] if trajectory.exists() else []
        tests = trial_dir / "verifier" / "test-stdout.txt"
        details[name] = {
            "steps": sum(s.get("source") == "agent" for s in steps) if steps else None,
            "tool_calls": sum(len(s.get("tool_calls") or []) for s in steps) if steps else None,
            "final_message": next((str(s["message"]) for s in reversed(steps) if s.get("source") == "agent"), ""),
            "tests": tests.read_text(errors="replace") if tests.exists() else "",
        }
# Tables: one row per task (results) and one per task phase (phases)
phases = pd.DataFrame(phases, columns=["task", "phase", "start", "end"])
for column in ("start", "end"):  # typed even when empty
    phases[column] = pd.to_datetime(phases[column], utc=True)
results = (
    pd.DataFrame(rows)
    .reindex(columns=["task", "reward", "error", "error_message", "tokens", "cost_usd", "agent_sec"])
    .set_index("task")
)
print(
    f"$ nemo --context {CONTEXT} jobs results download agent-eval-results --job <run>-task-<i> …   # {len(details)} tasks with a trial"
)

# Add task metadata and agent step counts; show one row per task
missing = sorted(set(results.index) - set(meta.index))
if missing:  # RUN_ID or COMPARE can name runs of other tasks
    meta = pd.concat([meta, task_metadata(missing)])
counts = pd.DataFrame.from_dict(details, orient="index").reindex(columns=["steps", "tool_calls"])
df = results.join(meta, how="left").join(counts, how="left")
df["reward"] = pd.to_numeric(df.reward)
df["solved"] = df.reward.fillna(0) >= 1
config = run_meta.get("target", {}).get("config", {})
AGENT_LABEL = f"{config.get('agent_name', 'agent')} · {config.get('agent_model_name') or 'no model'}"
display(
    df[["title", "language", "reward", "error", "agent_sec", "tokens", "steps", "tool_calls"]].sort_values("reward")
)

## 6. What happened

Headline numbers, then four views: when each task ran, where the agent's time went, how it did per language, and whether more tokens bought more solves.

In [ ]:
# Dashboard: headline numbers and 4 charts
# Chart colors and the counts the charts use
PASS, FAIL, GREY = "#009E73", "#D55E00", "#9e9e9e"
RAN = not phases.empty  # False when no task got as far as a sandbox
n, solved = len(df), int(df.solved.sum())
errored = int(df.error.notna().sum())
has_tokens, has_cost = df.tokens.notna().any(), df.cost_usd.notna().any()
# Nothing ran: say so instead of drawing empty charts
if not RAN:
    print(f"No task ran: {errored} of {n} ended before a sandbox started; see `error` in the table above.")
else:
    # Wall-clock vs the sum of task times
    t0, t1 = phases.start.min(), phases.end.max()
    task_spans = phases.groupby("task").agg(start=("start", "min"), end=("end", "max"))
    task_time = (task_spans.end - task_spans.start).dt.total_seconds()
    wall = (t1 - t0).total_seconds()

    def minutes(seconds):
        return f"{seconds / 60:.1f} min"

    # Headline numbers as tiles
    kpis = (
        [("Solved", f"{solved}/{n}"), ("Pass rate", f"{solved / n:.0%}")]
        + ([("Did not run", f"{errored}")] if errored else [])
        + [
            ("Wall-clock", minutes(wall)),
            ("Sum of task time", minutes(task_time.sum())),
            ("Parallel speedup", f"{task_time.sum() / wall:.1f}×"),
        ]
    )
    if has_cost:
        kpis += [
            ("Cost", f"${df.cost_usd.sum():.2f}"),
            ("Per solved task", f"${df.cost_usd.sum() / max(solved, 1):.2f}"),
        ]
    elif has_tokens:
        kpis += [
            ("Tokens", f"{df.tokens.sum() / 1e6:.1f}M"),
            ("Per solved task", f"{df.tokens.sum() / max(solved, 1) / 1e6:.1f}M"),
        ]
    display(
        HTML(
            '<div style="display:flex;gap:10px;flex-wrap:wrap;font-family:sans-serif">'
            + "".join(
                f'<div style="border:1px solid #ddd;border-radius:10px;padding:10px 16px"><div style="font-size:12px;color:#666">{k}</div>'
                f'<div style="font-size:22px;font-weight:600">{v}</div></div>'
                for k, v in kpis
            )
            + f"</div><p>{AGENT_LABEL}</p>"
        )
    )

    # Four charts in a 2×2 grid
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    order = task_spans.sort_values("start").index.tolist()

    # Timeline: one row per task, its four Harbor phases as minutes since the first sandbox started
    ax = axes[0, 0]
    shades = {"environment_setup": "#cfd8dc", "agent_setup": "#90a4ae", "verifier": "#37474f"}
    for y, task in enumerate(order):
        for row in phases[phases.task == task].itertuples():
            color = shades.get(row.phase) or (PASS if df.solved.get(task) else FAIL)
            ax.barh(
                y, (row.end - row.start).total_seconds() / 60, left=(row.start - t0).total_seconds() / 60, color=color
            )
        if pd.notna(df.error.get(task)):  # ended before the agent ran: say why next to its short bar
            ax.text(
                (task_spans.end[task] - t0).total_seconds() / 60 + 0.5,
                y,
                df.error[task],
                va="center",
                fontsize=7,
                color=FAIL,
            )
    ax.set_yticks(range(len(order)), order, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("minutes since the first sandbox started")
    ax.set_title("When each task ran")
    legend = [
        plt.Rectangle((0, 0), 1, 1, color=c)
        for c in (shades["environment_setup"], shades["agent_setup"], PASS, FAIL, shades["verifier"])
    ]
    ax.legend(
        legend,
        ["sandbox setup", "agent install", "agent (solved)", "agent (not solved)", "hidden tests"],
        fontsize=7,
        loc="lower right",
    )

    # Time per phase, per task
    ax = axes[0, 1]
    durations = (
        phases.assign(minutes=(phases.end - phases.start).dt.total_seconds() / 60)
        .pivot_table(index="task", columns="phase", values="minutes", aggfunc="sum")
        .fillna(0)
    )
    durations = durations.loc[durations.sum(axis=1).sort_values().index]
    left = pd.Series(0.0, index=durations.index)
    for phase in ("environment_setup", "agent_setup", "agent_execution", "verifier"):
        if phase not in durations:
            continue
        colors = (
            [PASS if df.solved.get(t) else FAIL for t in durations.index]
            if phase == "agent_execution"
            else shades[phase]
        )
        ax.barh(durations.index, durations[phase], left=left, color=colors)
        left += durations[phase]
    ax.tick_params(axis="y", labelsize=8)
    ax.set_xlabel("minutes")
    ax.set_title("Where each task's time went")

    # Pass rate by language, with counts
    ax = axes[1, 0]
    lang = df.groupby("language").solved.agg(["sum", "count"]).sort_values("count")
    ax.barh(lang.index, lang["count"], color="#e0e0e0")
    ax.barh(lang.index, lang["sum"], color=PASS)
    for y, (k, total) in enumerate(zip(lang["sum"], lang["count"])):
        ax.text(total + 0.05, y, f"{k}/{total}", va="center")
    ax.set_xlabel(f"tasks solved of tasks run (n={n}: a sample, not a benchmark score)")
    ax.set_title("Solved by language")

    # Tokens vs agent time
    ax = axes[1, 1]
    if has_tokens:
        ax.scatter(df.agent_sec / 60, df.tokens / 1e6, c=[PASS if s else FAIL for s in df.solved], s=60)
        for task, row in df.iterrows():
            ax.annotate(
                str(task)[:22],
                (row.agent_sec / 60, row.tokens / 1e6),
                fontsize=7,
                xytext=(4, 2),
                textcoords="offset points",
            )
        ax.set_xlabel("agent minutes")
        ax.set_ylabel("tokens (millions)")
        ax.set_title("Did more tokens buy a solve?")
    else:
        ax.text(0.5, 0.5, f"{AGENT_LABEL}: no model calls, no tokens", ha="center", va="center")
        ax.set_axis_off()
    fig.tight_layout()
    plt.show()

### Findings

Computed from this run; numbers only.

In [ ]:
# Findings, computed from the tables above
# Solved per language
lang = df.groupby("language").solved.agg(["sum", "count"]).sort_values("count")
findings = [
    f"**{AGENT_LABEL}** solved **{solved} of {n}** tasks ({solved / n:.0%}): "
    + ", ".join(f"{language} {k}/{t}" for language, (k, t) in lang[["sum", "count"]].iloc[::-1].iterrows())
    + "."
]
# Setup errors, apart from the agent's misses
if errored:
    findings.append(
        f"**{errored} of the {n - solved} unsolved tasks never reached the agent**: the trial ended with an error "
        f"before its hidden tests ran ({', '.join(f'`{t}` {e}' for t, e in df.error.dropna().items())}). "
        f"On the {n - errored} tasks it worked on, the agent solved {solved}."
    )
# Parallel speedup and the agent's share of task time
if RAN:
    findings.append(
        f"The tasks ran in **{minutes(wall)}** of wall-clock for {minutes(task_time.sum())} of task time: "
        f"**{task_time.sum() / wall:.1f}×** from running one Kubernetes Job per task."
    )
    agent_share = phases[phases.phase == "agent_execution"].pipe(lambda p: (p.end - p.start).dt.total_seconds().sum())
    findings.append(
        f"The agent itself used {agent_share / task_time.sum():.0%} of task time; "
        "the rest was sandbox setup, agent install and the hidden tests."
    )
# Tokens of unsolved vs solved tasks
worked = df[df.error.isna()]
if has_tokens and 0 < worked.solved.sum() < len(worked):
    hit, miss = worked[worked.solved].tokens.median(), worked[~worked.solved].tokens.median()
    findings.append(
        f"Unsolved tasks used **{miss / hit:.1f}×** the tokens of solved ones "
        f"(median {miss / 1e6:.2f}M vs {hit / 1e6:.2f}M)."
    )
# Cost, or tokens when the model reports no price
if has_cost:
    findings.append(
        f"Cost ${df.cost_usd.sum():.2f} in total, ${df.cost_usd.sum() / max(solved, 1):.2f} per solved task."
    )
elif has_tokens:
    findings.append(
        f"{df.tokens.sum() / 1e6:.1f}M tokens in total, {df.tokens.sum() / max(solved, 1) / 1e6:.1f}M per "
        "solved task (the gateway model reports no price, so tokens stand in for cost)."
    )
# The longest task
if RAN:
    longest = task_time.idxmax()
    findings.append(
        f"Longest task: `{longest}`, {minutes(task_time[longest])} end to end, "
        f"{minutes(df.agent_sec.get(longest) or 0)} of it the agent"
        + (f" ({int(df.steps[longest])} steps)." if pd.notna(df.steps.get(longest)) else ".")
    )
# Show the findings as a list
display(Markdown("\n".join(f"- {line}" for line in findings)))

### Why did a task fail?

`TASK` defaults to the first unsolved task the agent worked on. It shows what the agent said it did, next to what the hidden tests found. For a trial that ended with an error, it shows the error.

In [ ]:
# One unsolved task in detail: its phases, the agent's last message, the failing tests
# Pick a task: unsolved tasks the agent worked on come first
unsolved = df[~df.solved].sort_values("error", na_position="first")  # tasks the agent worked on come first
TASK = next(iter(unsolved.index), None)  # or any task name from the table above

# Its reward, steps, phase times and the agent's last message
if TASK is None:
    print("Every task was solved.")
else:
    d = details.get(TASK, {})
    tests = re.sub(r"\x1b\[[0-9;]*m", "", d.get("tests", ""))
    summary = r"--- FAIL|^FAIL|FAILED|✕|✗|AssertionError|Tests?:\s|\d+ failed|\[verifier\]"
    failing = [line for line in tests.splitlines() if re.search(summary, line)]
    task_phases = phases[phases.task == TASK].assign(
        minutes=lambda p: ((p.end - p.start).dt.total_seconds() / 60).round(1)
    )
    display(
        Markdown(
            f"#### `{TASK}`: {meta.title.get(TASK, '')}\n"
            f"- **Reward:** {'–' if pd.isna(df.reward[TASK]) else df.reward[TASK]} · **agent steps:** {d.get('steps') or '–'} · **tool calls:** {d.get('tool_calls') or '–'}\n"
            f"- **Phases (min):** " + ", ".join(f"{r.phase} {r.minutes}" for r in task_phases.itertuples()) + "\n\n"
            "**The agent's last message**\n\n> " + (d.get("final_message") or "(none)")[:800].replace("\n", "\n> ")
        )
    )
    # The error that ended the trial, or the hidden tests' summary lines
    if pd.notna(df.error[TASK]):
        message = re.sub(r"\x1b\[[0-9;]*m", "", str(df.error_message[TASK]))
        print(
            f"The trial ended with {df.error[TASK]} before the hidden tests ran:\n"
            + "\n".join(message.splitlines()[-12:])
        )
    else:
        print("Hidden tests (summary lines):\n" + "\n".join(failing[-20:] or tests.splitlines()[-20:]))

## 7. Next

Open the run in Studio: the Experiment's leaderboard has one row per run (dataset, agent, model, mean reward, tokens), and each trial's full trajectory as spans.

In [ ]:
# Studio links for this run, and the bundle's HTML report
# The Experiment's leaderboard and this run in Studio
print(f"Leaderboard: {EXPERIMENT_URL}\nThis run:    {EXPERIMENT_URL}/{RUN_ID}")
# The merged bundle's HTML report, when there is one
if (bundle / "report.html").exists():
    print(f"Report:      {bundle / 'report.html'}")

**Try next**

- **Another model on the same tasks:** set `MODEL`, keep `MODE = "taskset"`: nothing is published or built again, so the run starts in seconds.
- **Ceiling check:** `AGENT = "oracle"` applies each task's reference solution. A task that fails with oracle is an environment problem, not an agent one.
- **More tasks:** add names to `TASKS`; `--list-tasks` lists all 113.
- **More CPU or memory:** your admin's profile sets the ceilings; ask for a profile, then set `SANDBOX_PROFILE`.
- **Compare runs:** put 2+ run ids in `COMPARE` and run the cell below.

In [ ]:
# Compare the runs in COMPARE: reward per task, one column per run
if len(COMPARE) < 2:
    print("Set COMPARE to 2 or more run ids to compare them.")
else:
    # Each run's reward and error per task, from its merged bundle
    table, errors = {}, {}
    for run_id in COMPARE:
        folder = WORK / run_id
        if not (folder / "agent-eval-results").exists():
            nemo("evals", "runs", "download", run_id, "--workspace", WORKSPACE, "-o", str(folder), attempts=3)
        b = folder / "agent-eval-results"
        config = json.loads((b / "metadata.json").read_text())["target"]["config"]
        model = (config.get("agent_model_name") or "").removesuffix("-closed-book")
        label = f"{config.get('agent_name')}\n{model}\n{run_id[:12]}"
        table[label] = {
            s["task_id"].split("/")[-1]: next((o["value"] for o in s["outputs"] if o["name"] == "reward"), None)
            for s in read_jsonl(b / "scores.jsonl")
            if s["metric_type"] == "harbor_reward"
        }
        errors[label] = {t["task_id"].split("/")[-1] for t in read_jsonl(b / "trials.jsonl") if t.get("error")}
    # Heatmap: one row per task, one column per run
    grid = pd.DataFrame(table)
    fig, ax = plt.subplots(figsize=(3 + 2.4 * grid.shape[1], 0.45 * grid.shape[0] + 1.8))
    ax.imshow(
        grid.fillna(-1).values,
        cmap=plt.matplotlib.colors.ListedColormap([GREY, FAIL, PASS]),
        vmin=-1.5,
        vmax=1.5,
        aspect="auto",
    )
    ax.set_xticks(range(grid.shape[1]), grid.columns, fontsize=8)
    ax.set_yticks(range(grid.shape[0]), grid.index, fontsize=8)
    for y, task in enumerate(grid.index):
        for x, column in enumerate(grid.columns):
            v = grid.at[task, column]
            text = "–" if pd.isna(v) else "err" if task in errors[column] else f"{v:.0f}"
            ax.text(x, y, text, ha="center", va="center", color="white")
    ax.set_title("Reward per task (green 1, red 0, err = trial ended before the tests, grey not run)", fontsize=10)
    fig.tight_layout()
    plt.show()
    # Pass rate per run
    display(grid.mean().rename("pass rate").to_frame().T)